**Metric**
- A macro-averaged ROC-AUC metric (with the adjustment to skip classes with no true positive labels) for several important reasons:

1. Handling Class Imbalance
Equal Importance: Macro-averaging computes the ROC-AUC independently for each class and then averages the results. This approach ensures that every class is given equal weight—even those with few examples. In biodiversity datasets, where some species may be rare, this is crucial.

Avoiding Misleading Averages: By skipping classes without true positives, the metric prevents penalizing the model for not predicting classes that are absent or extremely underrepresented in the ground truth. This helps avoid distortions in the evaluation due to extreme class imbalance.

2. Threshold Independence
Continuous Performance Assessment: The ROC-AUC metric evaluates the model's ability to rank predictions correctly over a range of thresholds. This is beneficial in cases where the optimal threshold for decision-making is not known beforehand, as it provides a comprehensive view of model performance.

3. Robustness in Diverse Datasets
Applicability to Ecological Data: Given the complex and noisy nature of acoustic data in ecological monitoring, ROC-AUC offers a stable and reliable measure. It captures the trade-off between true positive and false positive rates without being affected by the exact decision threshold.

- Accuracy

**Predictions**  

For each row_id, we predict the probability that a given species was present. There is one column per species. Each row covers a 3-second window of audio.

**Training Data**  
- Species Classifiction
- Genus/Family classification
- Species Detection (class) = questionable!

Focus on this.
- Call type classification ( call or buzz(song))
- Captioning (prediction)

Audio Input- Audio encoder - Windowing and Librosa/Q-Former/CNN/RL - Audio Embeddings - Prediction

Audio Input: The raw audio is captured.

Audio Encoder: The audio is transformed into a more useful form (e.g., MFCCs, spectrogram).

Windowing: The signal is divided into smaller windows to analyze local features.

Q-Former: A transformer-based model processes the features to capture long-range dependencies.  

CNN:

Reinforcment Learning:

Audio Embeddings: The model outputs a fixed-length vector representation of the audio.

Prediction: The embeddings are used to make a final prediction (classification or regression).

The most common audio file formats include:

Lossy Formats (compressed with some quality loss)
- MP3 (.mp3)
Most common audio format; widely supported, small file sizes, suitable for streaming and sharing.

- AAC (.aac or .m4a)
Improved quality over MP3 at similar bitrates; commonly used by Apple (iTunes, Apple Music).

- OGG Vorbis (.ogg)
Open-source format with good compression and audio quality; popular on streaming platforms.

- WMA (.wma)
Developed by Microsoft; declining in popularity but still common on Windows platforms.

Lossless Formats (compressed without losing quality)
- FLAC (.flac)
Widely popular among audiophiles for archiving high-quality audio without losing fidelity; compatible with many modern devices.

- ALAC (.m4a)
Apple's lossless audio format; supported across Apple devices and increasingly popular among audiophiles.

Uncompressed Formats
- WAV (.wav)
High-quality, uncompressed audio format common in professional audio production and editing.

- AIFF (.aiff, .aif)
Uncompressed format used primarily on Apple systems and by audio professionals.

- Mono channel (1 channel):

Consistency; easier to handle, reduces dimensionality.

Most audio analysis algorithms expect mono inputs.

- Sample rate (44.1 kHz):

Standard across many audio ML libraries.

Balances data quality and processing speed.

- Bit depth (16-bit PCM):

Standard for good audio quality, sufficient for ML.

**audiomentation**     
Augment audio dat: creating slightly modified versions of your original audio clips to improve the robustness and generalization of your machine learning model.

Augmentation	Description
- AddGaussianNoise	Adds random noise to simulate real-world background
- TimeStretch	Speeds up or slows down the audio
- PitchShift	Shifts the pitch up or down (without changing speed)
- Shift	Moves the audio forward or backward in time
- ClippingDistortion	Simulates audio distortion by clipping the waveform
- Gain	Increases or decreases the volume
- HighPassFilter / LowPassFilter	Filters out low or high frequencies
- PolarityInversion	Flips waveform upside down (not audible but changes signal)

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pydub import AudioSegment
import wave
import math
import soundfile as sf
import librosa
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift, Shift
from pathlib import Path
from tqdm import tqdm
from IPython.display import Audio, display
import shutil
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from collections import defaultdict
from PIL import Image
import gc
import psutil

c:\Users\shang\Programs\anaconda3\Lib\site-packages\torch\utils\_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(


Defining functions

In [4]:
# a function to print the properties of an audio file

def f_get_properties(filename): 
    with wave.open(filename, 'rb') as wav_file:
        num_channels = wav_file.getnchannels()
        sample_rate = wav_file.getframerate()
        sample_width = wav_file.getsampwidth()
        num_frames = wav_file.getnframes()
        duration = num_frames / float(sample_rate)
        bit_depth = sample_width * 8  # Sample width is in bytes
    return num_channels, sample_rate, bit_depth, duration

In [ ]:
def f_convert_and_normalize_audio(
    dir_source="audio_input",
    dir_wav="audio_wav",
    dir_target="audio_output",
    target_sr=44100,
    subtype="PCM_16",
    mono=True
):
    """
    Converts audio files from multiple formats to WAV, then normalizes them
    to mono, 44.1kHz sample rate, and 16-bit PCM.
    Frees up space by deleting intermediate WAVs after normalization.
    """
    # Create directories if needed
   
    os.makedirs(dir_wav, exist_ok=True)
    os.makedirs(dir_target, exist_ok=True)

    # Step 1: Converting all formats to WAV.
    for file_name in os.listdir(dir_source):
        if file_name.lower().endswith((
            ".mp3", ".ogg", ".flac", ".aac", ".m4a", ".wma",
            ".aif", ".aiff", ".mp4", ".mpga", ".wav"
        )):
            input_path = os.path.join(dir_source, file_name)
            output_path = os.path.join(dir_wav, os.path.splitext(file_name)[0] + ".wav")

            try:
                audio = AudioSegment.from_file(input_path)
                audio.export(output_path, format="wav")
            except Exception as e:
                print(f"Failed to convert {file_name}: {e}")

    # Step 2: Normalizing WAVs to mono, 44.1kHz, 16-bit PCM
    for file_name in os.listdir(dir_wav):
        input_path = os.path.join(dir_wav, file_name)
        output_path = Path(dir_target) / file_name

        try:
            samples, sr = librosa.load(input_path, sr=target_sr, mono=mono)
            sf.write(output_path, samples, samplerate=target_sr, subtype=subtype)

            # Delete intermediate .wav to save space
            os.remove(input_path)

        except Exception as e:
            print(f"Could not normalize {file_name}: {e}")

    print("All audio files processed and saved to:", dir_target)

    # Clean up intermediary folders if empty
    try:
        os.rmdir(dir_wav)
        print(f"Removed temporary folder: {dir_wav}")
    except OSError:
        print(f"Could not remove {dir_wav} (not empty or in use).")


In [ ]:
def f_audio_clean(y, threshold_db=-35):
    """
    Determines if the audio chunk contains meaningful sound based on RMS energy.
    """
    if not isinstance(y, np.ndarray):
        y = np.array(y).astype(np.float32)
    rms = librosa.feature.rms(y=y)[0]
    db = librosa.amplitude_to_db(rms, ref=np.max)
    percent_below = np.mean(db < threshold_db)
    return percent_below < 0.9


In [ ]:
def f_split(input_folder="audio_output", duration=3):

    """
    Splits pre-normalized WAV files into chunks of fixed duration,
    padding the final chunk with silence if it's too short.
    Moves non-meaningful audio files to a _skipped folder.
    Deletes input files and folder after processing.

    """
    output_folder_split = f"{input_folder}_{duration}" # Create output folder name
    output_folder_skipped = f"{output_folder_split}_skipped" # Create skipped folder name
    os.makedirs(output_folder_split, exist_ok=True)
    os.makedirs(output_folder_skipped, exist_ok=True)

    chunk_length_ms = duration * 1000

    for audio_file in os.listdir(input_folder):
        if audio_file.lower().endswith(".wav"):
            input_path = os.path.join(input_folder, audio_file)
            audio = AudioSegment.from_wav(input_path)
            total_length_ms = len(audio)
            num_chunks = math.ceil(total_length_ms / chunk_length_ms)
            base_name = os.path.splitext(audio_file)[0]

            for i in range(num_chunks):
                start_ms = i * chunk_length_ms
                end_ms = start_ms + chunk_length_ms
                chunk_audio = audio[start_ms:end_ms]

                # Pad if needed
                if len(chunk_audio) < chunk_length_ms:
                    padding = AudioSegment.silent(duration=chunk_length_ms - len(chunk_audio))
                    chunk_audio += padding
                chunk_audio = chunk_audio[:chunk_length_ms]

                # Calculate time boundaries in seconds
                start_sec = int(start_ms / 1000)
                end_sec = int(start_sec + duration)

                chunk_filename = os.path.join(
                    output_folder_split,
                    f"{base_name}_{start_sec:03d}_{end_sec:03d}.wav"
                )

                # Export first, then check meaningfulness
                chunk_audio.export(chunk_filename, format="wav")
                y = np.array(chunk_audio.get_array_of_samples()).astype(np.float32)

                if not f_audio_clean(y):
                    os.rename(chunk_filename, os.path.join(output_folder_skipped, os.path.basename(chunk_filename)))
            # Delete the processed file
            os.remove(input_path)

    # Delete the input folder after all files are processed
    os.rmdir(input_folder)

    print(f"Split and cleaned audio into {output_folder_split}")
    print(f"Skipped audio saved in {output_folder_skipped}")
    print(f"Deleted folder: {input_folder}")

In [ ]:
def process_audio_files(input_folder, duration=3): 
    """
    Full audio processing:
    1. Convert and normalize to WAV format.
    2. Split into chunks of specified duration and filter silence.
    """
    f_convert_and_normalize_audio(dir_source=input_folder)
    f_split(input_folder="audio_output", duration=duration)

In [ ]:
def move_silent_chunks(folder_with_chunks, threshold_db=-35):
    """
    Moves silent chunks from a folder into a '_skipped' subfolder based on RMS threshold.
    """
    skipped_folder = f"{folder_with_chunks}_skipped"
    os.makedirs(skipped_folder, exist_ok=True)

    for file_name in os.listdir(folder_with_chunks):
        if file_name.lower().endswith(".wav"):
            file_path = os.path.join(folder_with_chunks, file_name)

            try:
                audio = AudioSegment.from_wav(file_path)
                y = np.array(audio.get_array_of_samples()).astype(np.float32)

                if not f_audio_clean(y, threshold_db=threshold_db):
                    os.rename(file_path, os.path.join(skipped_folder, file_name))
                    print(f"Moved to skipped: {file_name}")

            except Exception as e:
                print(f"Error processing {file_name}: {e}")

    print(f"Skipped audio saved in {skipped_folder}")

# Preprocessing data for the analysis

Converting audio files to format to .wav  
And resampling all WAVs to:
- Mono
- 44.1 kHz
- 16-bit PCM

Checking the properties of the files

In [ ]:
# Root folder containing audio subfolders
root_folder = "audio_3_sec"  # Replace with the path to your root folder containing subfolders of WAV files

# Traverse all folders and collect all WAV files
for root, dirs, files in os.walk(root_folder):
    for file in files:
        if file.lower().endswith(".wav"):
            filepath = os.path.join(root, file)

            # Get condition name (assumes folder name is the condition)
            condition = os.path.basename(root)

            # Get WAV properties using wave module
            num_channels, sample_rate, bit_depth, duration = get_properties(filepath)
            print(f"File: {file}")
            print(f"Number of Channels: {num_channels}")
            print(f"Sample Rate: {sample_rate} Hz")
            print(f"Bit Depth: {bit_depth}-bit")
            print(f"Duration: {duration:.2f} seconds")
            print("===============================")


## Defining which audio recordings could be used for "other" class

In [ ]:
df = pd.read_csv('data/data_20250330.csv')

# Create a new column to get unique identifiers for each row
df['Unique'] = df['ID'].astype(str) + '_' + df['Month'].astype(str) + '_' + df['Day'].astype(str) + '_' + df['Time'].astype(str)

df = df[df['Habitant'] == 1].reset_index(drop=True)  # Keep only rows where Habitant is 1

df = df[['Common name', 'Confidence', 'Unique', 'ID']]

df.info()

In [ ]:
# adding a check column to mark rows with "hume's warbler"
df['Check'] = (df['Common name'] == "hume's warbler").astype(int)

marked_uniques = df.loc[df['Check'] == 1, 'Unique'].unique()

# Fill out the 'Check' column based on marked uniques
df.loc[df['Unique'].isin(marked_uniques), 'Check'] = 1

# choosing the location where no "hume's warbler" was found
other = df[df['Check'] == 0]



In [ ]:
other = other[other['Confidence'] > 0.5] # adjust level as needed
other.info()

In [ ]:
(other['Common name'].value_counts(normalize=True)*100).round(3)

Define which unique audio recordings can be used

In [ ]:
samples = other.drop_duplicates(subset='Unique').reset_index(drop=True)  # Remove duplicates based on 'Unique' column

In [ ]:
samples['Common name'].nunique() 

In [ ]:
samples.to_csv('data/other.csv', index=False)  # Save the filtered samples to a new CSV file

Working with splitted audio set that is used for "other" classification only

Exploring random buzz and call sounds

In [ ]:
data_paths = {
    'buzz': 'audio_data/buzz/1.WAV',
    'song': 'audio_data/song/0.WAV'
} 


In [ ]:
# Checking the properties of the specified audio files in data_path
for condition, filepath in data_paths.items():
    num_channels, sample_rate, bit_depth, duration = f_get_properties(filepath)
    print(f"Condition: {condition}")

    filename = os.path.basename(filepath)

    print(f"File: {filename}")
    print(f"Number of Channels: {num_channels}")
    print(f"Sample Rate: {sample_rate} Hz")
    print(f"Bit Depth: {bit_depth}-bit")
    print(f"Duration: {duration:.2f} seconds")
    print("-------------------------------")


Let's play each audio sample

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    print(f"Name of the sample: {key}")
    display(Audio(x, rate=sr))

Visual inspection of waveform of each sample

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Generate time values for the x-axis
    time = librosa.times_like(x, sr=sr)
    
    plt.figure(figsize=(12, 4))
    plt.plot(time, x, label='Waveform', linewidth=2)
    plt.title(f'Waveplot of {key}')
    plt.xlabel('Time (s)')
    plt.ylabel('Amplitude')
    plt.legend()
    plt.show()

Zooming in on the data

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Zoom in on a specific range
    n0 = 400
    n1 = 800
    plt.figure(figsize=(12, 4))
    plt.plot(x[n0:n1])
    plt.title(f'Zoomed-in Waveform of {key}')
    plt.xlabel('Sample Index')
    plt.ylabel('Amplitude')
    plt.grid()
    plt.show()

Creating a spectrogram to visualize the frequency content of the signal over time  
The color intensity in the spectrogram represents the amplitude of different frequencies at different time points

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Compute the Short-Time Fourier Transform (STFT)
    X = librosa.stft(x)
    
    # Convert magnitude spectrogram to decibels
    Xdb = librosa.amplitude_to_db(abs(X))
    
    # Plot the spectrogram
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='hz')
    plt.colorbar()
    plt.title(f'Spectrogram of {key}')
    plt.show()


Displaying a spectrogram with a logarithmic frequency scale

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Compute the Short-Time Fourier Transform (STFT)
    X = librosa.stft(x)
    
    # Convert magnitude spectrogram to decibels
    Xdb = librosa.amplitude_to_db(abs(X))
    
    # Plot the spectrogram
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(Xdb, sr=sr, x_axis='time', y_axis='log')
    plt.colorbar()
    plt.title(f'Spectrogram of {key}')
    plt.show()

Visualization of Mel-frequency cepstral coefficients (MFCCs)  

MFCCs are coefficients representing the short-term power spectrum of a sound signal.  

The MFCCs capture the spectral characteristics of the audio signal and are particularly useful for capturing features related to human perception of sound.

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=44100)
    
    # Compute the mel spectrogram
    S = librosa.feature.melspectrogram(y=x, sr=sr, n_mels=128, fmax=8192)

    # Convert the mel spectrogram to MFCCs
    mfccs = librosa.feature.mfcc(S=librosa.power_to_db(S), n_mfcc=13)

    plt.figure(figsize=(12, 4))
    plt.title(f'MFCCs of {key}')
    librosa.display.specshow(mfccs, sr=sr, x_axis='time')
    plt.colorbar(format='%+2.0f dB')
    plt.show()

Generating a chromagram.  
Visual representation of the distribution of pitch content over time in the audio signal.  
The resulting plot is a useful tool for analyzing the harmonic content and tonal characteristics of the audio.

In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=None)
# Set the hop length
    hop_length = 12

# Compute the chromagram
    chromagram = librosa.feature.chroma_stft(y=x, sr=sr, hop_length=hop_length)

# Plot the chromagram
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(chromagram, x_axis='time', y_axis='chroma', hop_length=hop_length, cmap='coolwarm')
    plt.title(f"Chromagram of {key}")
    plt.colorbar()
    plt.show()


In [ ]:
for key, value in data_paths.items():
    x, sr = librosa.load(value, sr=None)

# Compute the chromagram
    chromagram = librosa.feature.chroma_cens(y=x)

# Plot the chromagram
    plt.figure(figsize=(12, 4))
    librosa.display.specshow(chromagram, x_axis='time', y_axis='chroma', cmap='coolwarm')
    plt.colorbar()
    plt.title('Chroma CENS')
    plt.show()

**Summary**:

Augmenting audio data   

In [ ]:
def f_audio_augment(
    source_dir,
    output_dir,
    max_per_class=100,
    augment=None
):
    """
    Augments pre-normalized, chunked 3-second .wav files until each class reaches `max_per_class`.

    Args:
        source_dir (str): Path to labeled class folders with 3-sec WAV files.
        output_dir (str): Where to save augmented WAVs.
        max_per_class (int): Max total number of segments (original + augmented) per class.
        augment (Compose or None): Audiomentations pipeline.
    """
    output_dir = f"{source_dir}_augmented" # Create output folder name
    os.makedirs(output_dir, exist_ok=True)
    class_counts = defaultdict(int)
    class_segments = defaultdict(list)

    # Default augmentation pipeline
    if augment is None:
        augment = Compose([
            AddGaussianNoise(min_amplitude=0.001, max_amplitude=0.015, p=0.7),
            TimeStretch(min_rate=0.95, max_rate=1.05, p=0.6),
            PitchShift(min_semitones=-1, max_semitones=1, p=0.3),
            Shift(min_shift=-0.1, max_shift=0.1, p=0.8)
        ])

    for file_path in Path(source_dir).rglob("*.wav"):
        label = file_path.parent.name
        samples, sample_rate = sf.read(file_path)

        class_counts[label] += 1
        class_segments[label].append((file_path.stem, samples, sample_rate))

    # Print class distribution BEFORE augmentation
    print("\n Original Class Distribution:")
    for label, count in class_counts.items():
        print(f" - {label:<10}: {count} segments")


    for label, segments in class_segments.items():
        aug_idx = 0
        while class_counts[label] < max_per_class:
            for stem, samples, sr in segments:
                if class_counts[label] >= max_per_class:
                    break
                augmented = augment(samples=np.asarray(samples, dtype=np.float32), sample_rate=sr)
                aug_name = f"{label}_{stem}_aug{aug_idx}.wav"
                augmented_subfolder = os.path.join(output_dir, f"{label}_augmented")
                os.makedirs(augmented_subfolder, exist_ok=True)

                out_path = os.path.join(augmented_subfolder, aug_name)
                sf.write(out_path, augmented, sr, subtype="PCM_16")
                class_counts[label] += 1
            aug_idx += 1

    print("\n Augmentation complete.")
    print(" Final Segment Counts Per Class:")
    for label, count in class_counts.items():
        print(f"{label:<10} --> {count} samples")

In [ ]:
f_audio_augment("audio_data")


 Original Class Distribution:
 - buzz      : 4 segments
 - song      : 84 segments

 Augmentation complete.
 Final Segment Counts Per Class:
buzz       --> 100 samples
song       --> 100 samples


In [ ]:
class SpectrogramEmbedder(nn.Module):
    def __init__(self, embedding_dim=128):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1)
        )
        self.fc = nn.Linear(64, embedding_dim)

    def forward(self, x):
        x = self.model(x)
        x = x.view(x.size(0), -1)
        return self.fc(x)



In [ ]:
def extract_combined_features_and_embeddings(
    input_folder,
    model,
    device,
    sr=44100,
    n_mfcc=13,
    n_mels=128,
    img_size=224
):
    transform = transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor()
    ])

    columns = (
        [f'chroma_{i}' for i in range(12)] +
        ['centroid', 'bandwidth', 'rolloff', 'zcr'] +
        [f'mfcc_{i}' for i in range(n_mfcc)] +
        [f'emb_{i}' for i in range(model.fc.out_features)] +
        ['filename']
    )

    all_data = []

    for file in os.listdir(input_folder):
        if not file.endswith(".wav"):
            continue

        file_path = os.path.join(input_folder, file)

        try:
            y, sr_actual = librosa.load(file_path, sr=sr, mono=True)

            # Audio features
            chroma = np.mean(librosa.feature.chroma_stft(y=y, sr=sr_actual), axis=1)
            centroid = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr_actual))
            bandwidth = np.mean(librosa.feature.spectral_bandwidth(y=y, sr=sr_actual))
            rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr_actual))
            zcr = np.mean(librosa.feature.zero_crossing_rate(y))
            mfcc = np.mean(librosa.feature.mfcc(y=y, sr=sr_actual, n_mfcc=n_mfcc), axis=1)

            features = np.concatenate([chroma, [centroid], [bandwidth], [rolloff], [zcr], mfcc])

            # Mel spectrogram to image (memory-only)
            mel = librosa.feature.melspectrogram(y=y, sr=sr_actual, n_mels=n_mels)
            mel_db = librosa.power_to_db(mel, ref=np.max)

            fig = plt.figure(figsize=(4, 4), dpi=100)
            ax = fig.add_axes([0, 0, 1, 1])
            librosa.display.specshow(mel_db, sr=sr_actual, cmap='gray')
            ax.axis('off')
            fig.canvas.draw()

            img = Image.frombytes('RGB', fig.canvas.get_width_height(), fig.canvas.tostring_rgb())
            plt.close(fig)

            # Pad and resize
            w, h = img.size
            max_dim = max(w, h)
            padded = Image.new("RGB", (max_dim, max_dim), color=(0, 0, 0))
            padded.paste(img, ((max_dim - w) // 2, (max_dim - h) // 2))
            resized = padded.resize((img_size, img_size), Image.LANCZOS)

            # To tensor and embed
            img_tensor = transform(resized).unsqueeze(0).to(device)
            with torch.no_grad():
                embedding = model(img_tensor).squeeze().cpu().numpy()

            # Combine all
            all_data.append(np.concatenate([features, embedding, [file]]))

        except Exception as e:
            print(f"Error processing {file}: {e}")

    df = pd.DataFrame(all_data, columns=columns)
    return df

In [ ]:
if __name__ == "__main__":
    input_folder = "path_to_wav_files"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = SpectrogramEmbedder(embedding_dim=128).to(device)
    model.eval()

    df_combined = extract_combined_features_and_embeddings(input_folder, model, device)
    df_combined.to_csv("combined_audio_dataset.csv", index=False)